# Week 5 — A knowledge graph of the course

The lecture gave a method for building a knowledge graph: competency questions first,
then the model they require, a review of the model, extraction, resolution, and an
idempotent load. This session applies the whole method once, on one example that
everyone shares: **this course**, built from its own documents.

You start with what you will have in Lab 2: a folder of documents and nothing else.
**Week 3's sample corpus** is the course's own material, the lecture and practical decks
and plans of Weeks 1 to 3, chunked into 183 chunks. Every node beyond the documents and
chunks comes from extraction, and every extracted relationship records the chunk that
states it, so that an answer built on it can cite its source.

You apply the same method to your own domain in **Lab 2**. `week5/ingest.py`, which
rebuilds this graph in one command, is the template for it.

**Before you start:** Neo4j running (`docker compose up -d` from the repository root),
and a Lightning AI key in `.env` as `LIGHTNING_API_KEY` (see `.env.example`). Parts 3,
4 and 7 are the only parts that call a model: twelve short calls.

In [ ]:
# Run this notebook from any folder. Week 5's modules live in week5/, and they reuse
# Week 2's model client (common/), Week 3's chunking, and Week 4's database helpers.
import sys, pathlib
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "requirements.txt").exists()), pathlib.Path.cwd().parent)
for _d in ("common", "week3", "week4", "week5"):
    sys.path.insert(0, str(_root / _d))
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

---
## Part 1 — Competency questions

A competency question is a question the graph must be able to answer. It comes first,
before any entity or relationship is chosen, and it should be a question that vector
retrieval cannot answer: one whose answer depends on how facts are connected, not on
any single chunk.

**Exercise 1.** Read each query against the model in Part 2. For each, fill in the
*Needs* column above, and name every relationship type the query traverses. A
relationship type that none of the five traverses is a candidate defect.
Part 5 runs these queries on a ten-chunk sample, Part 6 on the whole corpus.

In [ ]:
QUERIES = {
    "Q1": """
        // Which documents mention Chroma, and for which week?
        MATCH (w:Week)<-[:FOR_WEEK]-(d:Document)-[:HAS_CHUNK]->(c:Chunk)-[:MENTIONS]->(:Tool {name: 'Chroma'})
        RETURN d.name AS document, w.number AS week, count(c) AS chunks
        ORDER BY week, document
    """,
    "Q2": """
        // Which topics use Chroma, and where is that stated?
        MATCH (t:Topic)-[r:USES]->(:Tool {name: 'Chroma'})
        RETURN t.name AS topic, r.chunks AS stated_in
        ORDER BY topic
    """,
    "Q3": """
        // Which tools are taught in Week 3?
        MATCH (w:Week {number: 3})<-[:FOR_WEEK]-(d:Document)-[:HAS_CHUNK]->(c:Chunk)-[:MENTIONS]->(t:Tool)
        RETURN DISTINCT t.name AS tool, collect(d.name) AS documents
        ORDER BY t.name
    """,
    "Q4": """
        // Which tools appear in more than one week's material?
        MATCH (w:Week)<-[:FOR_WEEK]-(:Document)-[:HAS_CHUNK]->(:Chunk)-[:MENTIONS]->(t:Tool)
        WITH t, collect(DISTINCT w.number) AS weeks
        WHERE size(weeks) > 1
        RETURN t.name AS tool, weeks
        ORDER BY size(weeks) DESC, tool
    """,
}

---
## Part 2 — The extended model

The model is written down **once**, in `week5/kg_model.py`. The extraction schema and
the loader's check are both built from it, so they cannot disagree. Open the file and
read its docstring: it draws the text layer, and what extraction adds to it.

In [ ]:
from kg_model import ENTITY_TYPES, EXTRACTED_TYPES, STRUCTURAL_TYPES, ALIASES, TYPES

print("entity types extraction may return:  ", list(ENTITY_TYPES))
print("relationships extraction may return: ", EXTRACTED_TYPES)
print("relationships the loader writes:     ", STRUCTURAL_TYPES)
print(f"resolution: {len(ALIASES)} aliases, {len(TYPES)} settled types")

The text layer needs no API call. It comes from chunking the corpus exactly as Week 3
did (200 words, 40 of overlap), which is deterministic, so every chunk gets an id that
is the same on every machine: `<file>#<position>`.

In [ ]:
from kg_loader import read_chunks

chunks = read_chunks()
documents = sorted({c["source"] for c in chunks})
print(f"{len(chunks)} chunks from {len(documents)} documents")
for d in documents:
    print(f"  {sum(c['source'] == d for c in chunks):3}  {d}")
print("\nfirst chunk id:", chunks[0]["id"])

You should see 183 chunks from 12 documents. One of them,
`IDE-BCA001-heat-hold-cool-laplace-derivation.md`, is not about the course at all: it
is a mathematics derivation that found its way into the sample corpus. Extraction
should find nothing in it, which makes it a useful check.

### The design decisions

Read the model and answer these in a sentence each. They are the decisions the
lecture's method leaves open.

1. Why is `Document` connected to `Week` by a relationship, `FOR_WEEK`, rather than
   given a property `week: 3`? (Hint: which competency questions traverse it?)
2. Why is a document's `kind` (lecture, practical, plan) a **property**?
3. Two extracted names are one entity when they share a key: lower case and tidied,
   then looked up in a short alias table written by hand. Why is that table written
   by reading the extraction's output, rather than by asking the model which names
   mean the same thing?
4. Why does `r.chunks` hold each chunk id **once**, rather than one relationship per
   chunk?

The constraints are in `week5/constraints.cypher`: one uniqueness constraint per key,
the only kind Community edition has.

---
## Part 3 — A schema the model must follow

In Week 2 you first asked for JSON in the prompt, then passed a schema with `schema=`
and parsed the result. Extraction in Part 4 depends entirely on that mechanism, so this
part looks at it more closely, on questions that have nothing to do with graphs.

A schema is a Pydantic class. Its fields, and their types, are the shape of the answer.
`Literal` lists the only values a field may take.

In [ ]:
import json
from typing import Literal
from pydantic import BaseModel, ValidationError

class Message(BaseModel):
    category: Literal["setup", "concept", "deadline", "other"]
    urgent: bool
    summary: str
    short_summary: str
    

print(json.dumps(Message.model_json_schema(), indent=1))

This JSON Schema is what `llm.chat` sends with the request. Find three things in it:
`required` lists every field; `urgent` must be a boolean; and `category` is an `enum`,
the `Literal` written as a closed list of four values. The model is asked in strict mode,
so it can produce nothing outside this shape.

In [ ]:
from llm_client import LLMClient

llm = LLMClient(provider="lightning")
message = "I really need to know ASAP when Lab 1 has deadline"

result = llm.chat(message, system="Classify the student's message.", schema=Message)
print("text:  ", result.text)
print("parsed:", repr(result.parsed))
print("urgent is a", type(result.parsed.urgent).__name__)

A capital letter, a value that is not a boolean, a missing field: each is refused. So
there are two layers. Strict mode makes the model write only what the schema allows,
and the parse checks it again on arrival.

Compare asking for the same shape in the prompt alone, as in Week 2:

In [ ]:
ask = ("\n\nAnswer in JSON with the fields category (setup, concept, deadline or other), "
       "urgent (true or false) and summary.")
result = llm.chat(message + ask)
print(result.text)
try:
    print("\nparsed:", Message.model_validate_json(result.text))
except ValidationError as exc:
    print("\nrefused:", exc.errors()[0]["msg"])

---
## Part 4 — Extraction on a sample

Extraction reads chunks and returns what they state, in the shape the model declares.
Its schema is built exactly as in Part 3: three classes in `week5/extraction.py`,
`Mention`, `Relation` and `Extraction`. Their `type` fields are `Literal` types whose
values are read from `kg_model.py`, so the model **cannot** return a type the design
does not declare. Every mention and every relationship carries the id of the chunk it
came from, because that id is what an answer will later cite.

In [ ]:
from kg_model import ENTITY_TYPES
from extraction import Extraction, EntityType, instruction

print("tuple(ENTITY_TYPES):", tuple(ENTITY_TYPES))
print("EntityType:         ", EntityType)
print(json.dumps(Extraction.model_json_schema()["$defs"]["Mention"], indent=1))

`tuple()` of a dictionary gives its keys, so `Literal[tuple(ENTITY_TYPES)]` in
`extraction.py` is the same as writing `Literal["Topic", "Tool", "Deliverable"]`, and
the schema shows it as an `enum`. A student whose model declares `Person` and
`Institution` gets those two values here, from their own `kg_model.py`, without editing
`extraction.py`.

The instruction is plain text. Read it: it asks for relationships the chunk **states**,
not ones that are merely plausible, and defines each entity type in one line.

In [ ]:
print(instruction())

The sample is ten chunks, chosen to include chunks that state relationships, chunks
that tempt the model into wrong ones (a pipeline diagram, a table of parts, the course
stack, the list of week titles, a price table, an example about Einstein), and the one
chunk that is not about the course. Read them first; your own reading is the reference.

In [ ]:
from kg_loader import SAMPLE_IDS

sample = [c for c in chunks if c["id"] in SAMPLE_IDS]
for c in sample:
    print(f"\n===== {c['id']}\n{c['text'][:700]}{' ...' if len(c['text']) > 700 else ''}")

Now extract the sample twice, with the same instruction both times. Three chunks go
into each call, so each run is four calls.

**Which model.** The free Gemini tier allows only **20 generation requests per day**
per model (measured 2026-10-04), shared by everything you run that day, so this course
extracts with the second provider, Lightning AI. If you have no Lightning key, use
`provider="gemini"`: this notebook's twelve calls are within the day's twenty, but a full
extraction of your own corpus in Lab 2 will not be.

In [ ]:
from llm_client import LLMClient
from extraction import extract

llm = LLMClient(provider="lightning")
print("extracting with", llm.model)
first = extract(llm, sample)
second = extract(llm, sample)

In [ ]:
def by_chunk(results, chunk_id):
    names = sorted({f"{m['name']} ({m['type']})" for m in results["mentions"]
                    if m["chunk"] == chunk_id})
    rels = sorted(f"{r['source']} -{r['type']}-> {r['target']}" for r in results["relations"]
                  if r["chunk"] == chunk_id)
    return names, rels

for c in sample:
    print(f"\n===== {c['id']}")
    for label, res in (("first run ", first), ("second run", second)):
        names, rels = by_chunk(res, c["id"])
        print(f"  {label}  mentions:      {names}")
        print(f"  {'':10}  relationships: {rels}")
print("\nmentions citing a chunk that was not in the call (dropped):",
      first["dropped_unknown_chunk"], "/", second["dropped_unknown_chunk"])

Three things to look for.

- **Names drift between runs.** The same chunks and the same instruction give different
  names for the same thing. Nothing tells the model which names already exist, and in
  Lab 2 nothing could. That is why resolution, in Part 5, exists.
- **Whatever looks like a name becomes an entity.** Slide titles, the titles of weeks,
  the people in an example about Einstein, the functions of a mathematics derivation:
  extraction returns what the text contains, whether or not it is what the course is
  about.
- **A relationship type invites forcing.** The stages of a pipeline diagram, the parts
  of a table, the layers of the course stack can all come back as `BUILDS_ON` or
  `ASSESSES`. A closed schema prevents undeclared types; it does not prevent a wrong
  use of a declared one.

The same extraction, run once over the whole corpus for this course, is saved in
`extractions/course_corpus.json`. Its answers for your sample are below. Your runs will
differ from it, and from your neighbour's: extraction is not deterministic.

---
## Part 5 — Resolution and loading

Extraction returns names, and resolution decides two things about them: which names are
**one entity**, and what **type** it is. `resolve_results` groups names that share a
key: lower case, tidied, then looked up in `ALIASES` in `kg_model.py`. The type is the
one the model gave, unless `TYPES` settles it. Each entity is shown under the spelling
the model used most often.

Anything resolution cannot settle is reported, not guessed: names still returned under
two types, and relationships whose end then has the wrong type. A wrong merge puts a
false fact into the graph; a missed one only leaves two nodes where there should be one.

In [ ]:
# Re-read kg_model.py and kg_loader.py, so that running this cell again after you edit
# ALIASES or TYPES (Exercise 3) uses your new entries.
import importlib, kg_model, kg_loader
importlib.reload(kg_model); importlib.reload(kg_loader)
from kg_loader import resolve_results, load_all

resolved = resolve_results(first)
print(f"{len(resolved['entities'])} entities, {len(resolved['mentions'])} mentions, "
      f"{len(resolved['relations'])} relationships")
print("entities with several spellings:", resolved["merged"])
print("names under two types:", resolved["type_conflicts"])
print("relationships dropped for a wrong type:",
      [f"{r['source']} -{r['type']}-> {r['target']}" for r in resolved["dropped_relations"]])

**Exercise 3.** Read the entities with several spellings and the names under two types.
Which names in your two runs are one entity that the key did not merge (an entry for
`ALIASES`), and which type conflict can you settle (an entry for `TYPES`)? Read the
comment above `ALIASES` first. Week titles such as "Embeddings and semantic search"
name **weeks**; merging one into a topic would be a false fact.

The course corpus is in English, so lower-casing is enough here. A Ukrainian corpus in
Lab 2 also needs **lemmatised** keys, so that cases of one name merge, and an
inspection of proper names, on which lemmatisation is unreliable: measured
2026-10-04, «Іван Франко» lemmatises to `іван франко` but «Івана Франка» to
`іван франк`. Ask the model for the nominative form first.

### Load the sample

The loader writes exactly the `resolved` you inspected above: resolution is a step of its
own, and the load takes its result. The load follows the lecture's order: constraints, then the text layer, then the
entities, the relationships between them, and `MENTIONS`. Every statement merges on
a key, so loading twice must change nothing, **including** the lists of chunk ids.

In [ ]:
from graph_db import connect, run, counts, breakdown

driver = connect()
run(driver, "MATCH (n) DETACH DELETE n")          # start from an empty database

def snapshot():
    ids = run(driver, "MATCH ()-[r]->() WHERE r.chunks IS NOT NULL "
                      "RETURN sum(size(r.chunks)) AS n")[0]["n"]
    return {**counts(driver), "chunk ids on relationships": ids}

load_all(driver, sample, resolved)
once = snapshot()
load_all(driver, sample, resolved)
twice = snapshot()
print("after one load: ", once)
print("after two loads:", twice)
print("unchanged:      ", once == twice)

`load_all` checks every relationship type against `kg_model.py` before writing it into
a query. A type cannot be a query **parameter** in Cypher 5, so it becomes part of the
query text, and the model's output is untrusted input. Try it:

In [ ]:
for q, text in QUERIES.items():
    rows = run(driver, text)
    print(f"{q}: {len(rows)} rows", rows[:3])

---
## Part 6 — The whole corpus

Load the reference extraction of all 183 chunks. It costs no API call: it is a file.

In [ ]:
from extraction import load_results

# The reference extraction of all 183 chunks, saved to a file: no API call.
reference = load_results(_root / "week5" / "extractions" / "course_corpus.json", chunks)
run(driver, "MATCH (n) DETACH DELETE n")
resolved_all = resolve_results(reference)
load_all(driver, chunks, resolved_all)
print(f"{len(resolved_all['entities'])} entities, {len(resolved_all['mentions'])} mentions, "
      f"{len(resolved_all['relations'])} relationships; {len(resolved_all['merged'])} entities had "
      f"several spellings, {len(resolved_all['type_conflicts'])} names came under two types")
breakdown(driver)

You should see 12 `Document` and 183 `Chunk` nodes, 3 `Week` nodes, and 388 entities:
283 topics, 82 tools and 23 deliverables, reached by 614 `MENTIONS`. Extraction added
145 relationships: 76 `USES`, 49 `BUILDS_ON` and 20 `ASSESSES`, each with the chunks
that state it.

Most entities are thin. Count how many are mentioned in at most one chunk:

---
## Part 7 — Week 4's question, asked of this graph

Week 4 ended with *"What do I need to know before I write an agent that calls tools?"*:
a vector search found the starting topic, a traversal collected what it builds on, and
`ask` answered from those facts with Week 3's grounded instruction. Ask the graph you
built today the same question, in the same three steps.

This graph has no topic vectors, but it can have the chunk vectors of Week 3, copied
from Chroma. So the
search finds the chunks nearest to the question, `MENTIONS` leads to the topics they
discuss, and `BUILDS_ON` to what those topics build on, each fact with the chunk that
states it. `ask` uses the provider in your `.env`, as in Week 4: two calls and one
embedding.

In [ ]:
sys.path.insert(0, str(_root / "week2"))
from context_lab import ask
from embedding_client import EmbeddingClient
from graph_vectors import vector_search

# Week 3's grounded instruction, unchanged.
SYSTEM = (
    "You answer questions using only the sources you are given. If the sources do "
    "not answer the question, reply exactly: not stated. Never fill a gap from your "
    "own knowledge."
)
TASK = "Answer the question using only the SOURCES above."
FORMAT = "Two sentences maximum. Put the source number in square brackets after each claim."

QUESTION = "What do I need to know before I write an agent that calls tools?"

# The search needs the chunk vectors on the Chunk nodes. Part 6 loaded the graph without
# them; copy them from Week 3's Chroma store, as ingest.py does. No embedding call.
from kg_loader import copy_vectors
print(copy_vectors(driver, chunks), "chunk vectors copied from Chroma")

# 1. Search: the three chunks nearest in meaning to the question.
hits = vector_search(driver, EmbeddingClient(), QUESTION, index="chunk_embeddings", k=3)
if not hits:
    raise SystemExit("No chunk vectors to search: see Troubleshooting, `vectors: 0 of 183`.")
for h in hits:
    print(f"search landed on: {h['node']['id']}  (score {h['score']:.4f})")

# 2. Traverse: the topics those chunks mention, and what each builds on.
rows = run(driver, '''
    UNWIND $ids AS id
    MATCH (:Chunk {id: id})-[:MENTIONS]->(t:Topic)-[r:BUILDS_ON]->(p:Topic)
    RETURN DISTINCT t.name AS a, p.name AS b, r.chunks AS chunks ORDER BY a, b
''', ids=[h["node"]["id"] for h in hits])

# 3. Answer: the facts become numbered sources for the same grounded prompt.
facts = [f"{r['a']} builds on {r['b']}. (stated in {', '.join(r['chunks'])})" for r in rows]
for i, fact in enumerate(facts, 1):
    print(f"[{i}] {fact}")
result = ask(QUESTION, system=SYSTEM, task=TASK, fmt=FORMAT,
             sources=facts, show_prompt=False, label="search, traverse, answer")

In [ ]:
# The same three chunks, given to the model directly, as Week 3 would.
result = ask(QUESTION, system=SYSTEM, task=TASK, fmt=FORMAT,
             sources=[h["node"]["text"] for h in hits], show_prompt=False,
             label="chunks as sources")

Compare the two answers, and both with Week 4's. From the graph, the answer names what
tool calling builds on, and every claim leads to a fact and every fact to its chunk. From
the chunks alone, it is vaguer. Against Week 4 it gives up one thing: which week teaches
each prerequisite, since that was a hand-written relationship this graph does not have.

Follow one citation to its text before trusting the answer:
`[c["text"] for c in chunks if c["id"] == "week-02-practical-plan.md#4"]`. The chunk says
that *the difference between a prompt and a configuration* is the prerequisite for tool
calling; extraction made two prerequisites of it, and the model reported both. The
grounded instruction keeps the model to its sources. It cannot make the sources right.

Here a search chose the starting chunks and the graph took over from there. Choosing the
starting point well, and choosing which facts reach the model and how many, is the
subject of Week 6.

In [ ]:
driver.close()